# EXP-037 | Stacking (Level-2 Meta-Learner)

단순 가중 평균 대신 메타 모델이 최적 조합을 **학습**으로 찾는 방식.

| Level | 모델 | 피처 |
|---|---|---|
| Base (L1) | LGB-GBDT, CAT, XGB | FE-v1 (85개) |
| Base (L1) | LGB-GBDT, CAT, XGB | FE-v2 + TE + ITE |
| Meta (L2) | LightGBM | L1 OOF 예측값 6개 |

각 fold에서 L1 모델을 학습하고 OOF 예측을 메타 피처로 수집 → L2 모델이 최적 조합 학습.

| 기준선 | EXP-032 OOF 0.74071 / 제출 0.74219 |

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from datetime import date
from scipy.stats import rankdata

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, recall_score, precision_score, accuracy_score
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
import xgboost as xgb

from src.preprocessing import preprocess

warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/raw')
OUT_DIR  = Path('../data/submissions')
DOCS_DIR = Path('../docs')
TARGET   = '임신 성공 여부'
SEED     = 42
N_FOLDS  = 5
EXP_NO   = 37
AUTHOR   = '조여진'
CV_STR   = f'Stratified {N_FOLDS}-Fold'

train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
sub   = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print(f'train: {train.shape}  /  test: {test.shape}')

train: (256351, 69)  /  test: (90067, 68)


## 1. 피처 준비

In [2]:
def add_pre_encode_features(df):
    df = df.copy()
    df['기증_난자_여부'] = (df['난자 출처'] == '기증 제공').astype(int)
    df['기증_정자_여부'] = df['정자 출처'].isin(['기증 제공', '배우자 및 기증 제공']).astype(int)
    return df

def add_derived_features_v1(df):
    df = df.copy()
    eps = 1e-6
    df['수정률']    = df['총 생성 배아 수']   / (df['혼합된 난자 수'] + eps)
    df['이식률']    = df['이식된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['저장률']    = df['저장된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['ICSI_비율'] = df['미세주입된 난자 수'] / (df['혼합된 난자 수'] + eps)
    df['배아_발달일']    = df['배아 이식 경과일'] - df['난자 혼합 경과일']
    df['신선_시술_여부']  = df['수집된 신선 난자 수'].notna().astype(int)
    male_cols   = ['남성 주 불임 원인','남성 부 불임 원인','불임 원인 - 남성 요인']
    female_cols = ['여성 주 불임 원인','여성 부 불임 원인','불임 원인 - 난관 질환',
                   '불임 원인 - 배란 장애','불임 원인 - 자궁내막증','불임 원인 - 자궁경부 문제']
    couple_cols = ['부부 주 불임 원인','부부 부 불임 원인']
    sperm_cols  = ['불임 원인 - 정자 농도','불임 원인 - 정자 운동성',
                   '불임 원인 - 정자 형태','불임 원인 - 정자 면역학적 요인']
    all_cause   = male_cols + female_cols + couple_cols + sperm_cols + ['불명확 불임 원인']
    df['남성_불임_합계']      = df[male_cols].sum(axis=1)
    df['여성_불임_합계']      = df[female_cols].sum(axis=1)
    df['부부_불임_합계']      = df[couple_cols].sum(axis=1)
    df['정자_문제_합계']      = df[sperm_cols].sum(axis=1)
    df['총_불임원인_수']      = df[all_cause].sum(axis=1)
    df['임신시도기록_있음']    = df['임신 시도 또는 마지막 임신 경과 연수'].notna().astype(int)
    df['신선_난자_저장_있음']  = (df['저장된 신선 난자 수'] > 0).astype(int)
    df['나이_시술횟수_상호작용'] = df['시술 당시 나이'] * df['총 시술 횟수']
    return df

def add_derived_features_v2(df):
    df = add_derived_features_v1(df)
    eps = 1e-6
    df['임신_성공률']   = df['총 임신 횟수'] / (df['총 시술 횟수'] + eps)
    df['시술_실패_횟수'] = (df['총 시술 횟수'] - df['총 임신 횟수']).clip(lower=0)
    egg_count = df['수집된 신선 난자 수'].fillna(-1)
    df['최적_난자수']    = ((egg_count >= 5) & (egg_count <= 15)).astype(int)
    df['나이_이식배아수'] = df['시술 당시 나이'] * df['이식된 배아 수'].fillna(0)
    return df

TE_COLS   = ['시술 시기 코드','시술 유형','특정 시술 유형','배란 유도 유형',
             '난자 출처','정자 출처','시술 당시 나이','총 시술 횟수','총 임신 횟수']
ITE_PAIRS = [('시술 당시 나이','시술 유형'),('시술 당시 나이','특정 시술 유형'),
             ('시술 당시 나이','난자 출처'),('시술 당시 나이','배란 유도 유형'),
             ('시술 유형','총 시술 횟수'),('난자 출처','시술 유형')]
TE_ALPHA, ITE_ALPHA = 10, 20

train_fe = add_pre_encode_features(train)
test_fe  = add_pre_encode_features(test)
X_base_tr, X_base_te = preprocess(train_fe, test_fe)

X_v1_train = add_derived_features_v1(X_base_tr)
X_v1_test  = add_derived_features_v1(X_base_te)
X_v2_train = add_derived_features_v2(X_base_tr)
X_v2_test  = add_derived_features_v2(X_base_te)

TE_COLS   = [c for c in TE_COLS   if c in X_v2_train.columns]
ITE_PAIRS = [(c1,c2) for c1,c2 in ITE_PAIRS
             if c1 in X_v2_train.columns and c2 in X_v2_train.columns]

y_train = train[TARGET]
neg_pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f'FE-v1: {X_v1_train.shape[1]}개  /  FE-v2: {X_v2_train.shape[1]}개')
print(f'TE: {len(TE_COLS)}개  /  ITE: {len(ITE_PAIRS)}쌍')

FE-v1: 85개  /  FE-v2: 89개
TE: 9개  /  ITE: 6쌍


## 2. Level-1 Base 모델 파라미터

In [3]:
# ── FE-v1 계열 (EXP-020 파라미터) ──────────────────────────────────────────
LGB_V1 = dict(
    objective='binary', metric='auc', verbosity=-1, seed=SEED,
    num_threads=-1, is_unbalance=True, bagging_freq=1,
    learning_rate=0.035425966303284824,
    num_leaves=266, max_depth=5, min_child_samples=166,
    feature_fraction=0.5346439126449233,
    bagging_fraction=0.7122309235479091,
    lambda_l1=9.901034988600228,
    lambda_l2=2.213951873239442,
    min_gain_to_split=0.11418176854933762,
)
CAT_V1 = dict(
    iterations=2000, loss_function='Logloss', eval_metric='AUC',
    auto_class_weights='Balanced', random_seed=SEED,
    thread_count=-1, verbose=False, early_stopping_rounds=100,
    learning_rate=0.018758723768855998, depth=6,
    l2_leaf_reg=9.189608434163782, min_data_in_leaf=19,
    subsample=0.8170921295501483, colsample_bylevel=0.6936810336930781,
)
XGB_V1 = dict(
    n_estimators=2000, scale_pos_weight=neg_pos_ratio,
    eval_metric='auc', tree_method='hist', random_state=SEED,
    n_jobs=-1, verbosity=0, early_stopping_rounds=100,
    learning_rate=0.05520069867907647, max_depth=4, min_child_weight=59,
    subsample=0.7663066457187595, colsample_bytree=0.6581836436885355,
    reg_alpha=8.692038126211928, reg_lambda=0.23932562420374562,
)

# ── FE-v2+TE+ITE 계열 (EXP-032 파라미터 동일) ──────────────────────────────
LGB_V2 = {**LGB_V1}   # 같은 파라미터, 피처만 다름
CAT_V2 = {**CAT_V1}
XGB_V2 = {**XGB_V1}

# ── Level-2 Meta 모델 ───────────────────────────────────────────────────────
META_PARAMS = dict(
    objective='binary', metric='auc', verbosity=-1, seed=SEED,
    num_threads=-1, is_unbalance=True,
    learning_rate=0.05,
    num_leaves=8,        # 메타 피처 6개 → 과적합 방지 위해 작게
    max_depth=3,
    min_child_samples=500,
    lambda_l1=1.0,
    lambda_l2=1.0,
    feature_fraction=1.0,
    bagging_fraction=0.8,
    bagging_freq=1,
)

BASE_MODEL_NAMES = ['LGB_v1', 'CAT_v1', 'XGB_v1', 'LGB_v2', 'CAT_v2', 'XGB_v2']
N_BASE = len(BASE_MODEL_NAMES)
print(f'Base 모델: {N_BASE}개  →  {BASE_MODEL_NAMES}')
print(f'Meta 모델: LGB (num_leaves=8, max_depth=3)')

Base 모델: 6개  →  ['LGB_v1', 'CAT_v1', 'XGB_v1', 'LGB_v2', 'CAT_v2', 'XGB_v2']
Meta 모델: LGB (num_leaves=8, max_depth=3)


## 3. Level-1 학습 — OOF 메타 피처 생성

In [4]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# OOF 메타 피처 (train용)
oof_meta  = np.zeros((len(X_v1_train), N_BASE))
# Test 메타 피처 (fold 평균)
test_meta = np.zeros((len(X_v1_test),  N_BASE))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_v1_train, y_train), 1):
    print(f'\n── Fold {fold}/{N_FOLDS} ──────────────────────────────────')
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    global_mean = y_tr.mean()

    # ── FE-v1 슬라이스 ──
    X_tr_v1  = X_v1_train.iloc[tr_idx]
    X_val_v1 = X_v1_train.iloc[val_idx]

    # ── FE-v2 + TE 슬라이스 ──
    X_tr_v2  = X_v2_train.iloc[tr_idx].copy()
    X_val_v2 = X_v2_train.iloc[val_idx].copy()
    X_te_v2  = X_v2_test.copy()
    for col in TE_COLS:
        grp = y_tr.groupby(X_tr_v2[col])
        sm  = (grp.count() * grp.mean() + TE_ALPHA * global_mean) / (grp.count() + TE_ALPHA)
        tc  = f'{col}_te'
        X_tr_v2[tc]  = X_tr_v2[col].map(sm).fillna(global_mean)
        X_val_v2[tc] = X_val_v2[col].map(sm).fillna(global_mean)
        X_te_v2[tc]  = X_te_v2[col].map(sm).fillna(global_mean)

    # ── FE-v2 + TE + ITE (XGB_v2용) ──
    X_tr_xgb  = X_tr_v2.copy()
    X_val_xgb = X_val_v2.copy()
    X_te_xgb  = X_te_v2.copy()
    for col1, col2 in ITE_PAIRS:
        key_tr  = X_tr_xgb[col1].astype(str)  + '_' + X_tr_xgb[col2].astype(str)
        key_val = X_val_xgb[col1].astype(str) + '_' + X_val_xgb[col2].astype(str)
        key_te  = X_te_xgb[col1].astype(str)  + '_' + X_te_xgb[col2].astype(str)
        grp = y_tr.groupby(key_tr)
        sm  = (grp.count() * grp.mean() + ITE_ALPHA * global_mean) / (grp.count() + ITE_ALPHA)
        ic  = f'{col1}_{col2}_ite'
        X_tr_xgb[ic]  = key_tr.map(sm).fillna(global_mean)
        X_val_xgb[ic] = key_val.map(sm).fillna(global_mean)
        X_te_xgb[ic]  = key_te.map(sm).fillna(global_mean)

    fold_aucs = []

    # [0] LGB_v1
    ds_tr  = lgb.Dataset(X_tr_v1, label=y_tr)
    ds_val = lgb.Dataset(X_val_v1, label=y_val, reference=ds_tr)
    m = lgb.train(LGB_V1, ds_tr, num_boost_round=2000, valid_sets=[ds_val],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])
    oof_meta[val_idx, 0]  = m.predict(X_val_v1)
    test_meta[:, 0]      += m.predict(X_v1_test) / N_FOLDS
    fold_aucs.append(roc_auc_score(y_val, oof_meta[val_idx, 0]))

    # [1] CAT_v1
    m = CatBoostClassifier(**CAT_V1)
    m.fit(X_tr_v1, y_tr, eval_set=Pool(X_val_v1, y_val), use_best_model=True)
    oof_meta[val_idx, 1]  = m.predict_proba(X_val_v1)[:, 1]
    test_meta[:, 1]      += m.predict_proba(X_v1_test)[:, 1] / N_FOLDS
    fold_aucs.append(roc_auc_score(y_val, oof_meta[val_idx, 1]))

    # [2] XGB_v1
    m = xgb.XGBClassifier(**XGB_V1)
    m.fit(X_tr_v1, y_tr, eval_set=[(X_val_v1, y_val)], verbose=False)
    oof_meta[val_idx, 2]  = m.predict_proba(X_val_v1)[:, 1]
    test_meta[:, 2]      += m.predict_proba(X_v1_test)[:, 1] / N_FOLDS
    fold_aucs.append(roc_auc_score(y_val, oof_meta[val_idx, 2]))

    # [3] LGB_v2
    ds_tr  = lgb.Dataset(X_tr_v2, label=y_tr)
    ds_val = lgb.Dataset(X_val_v2, label=y_val, reference=ds_tr)
    m = lgb.train(LGB_V2, ds_tr, num_boost_round=2000, valid_sets=[ds_val],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])
    oof_meta[val_idx, 3]  = m.predict(X_val_v2)
    test_meta[:, 3]      += m.predict(X_te_v2) / N_FOLDS
    fold_aucs.append(roc_auc_score(y_val, oof_meta[val_idx, 3]))

    # [4] CAT_v2
    m = CatBoostClassifier(**CAT_V2)
    m.fit(X_tr_v2, y_tr, eval_set=Pool(X_val_v2, y_val), use_best_model=True)
    oof_meta[val_idx, 4]  = m.predict_proba(X_val_v2)[:, 1]
    test_meta[:, 4]      += m.predict_proba(X_te_v2)[:, 1] / N_FOLDS
    fold_aucs.append(roc_auc_score(y_val, oof_meta[val_idx, 4]))

    # [5] XGB_v2
    m = xgb.XGBClassifier(**XGB_V2)
    m.fit(X_tr_xgb, y_tr, eval_set=[(X_val_xgb, y_val)], verbose=False)
    oof_meta[val_idx, 5]  = m.predict_proba(X_val_xgb)[:, 1]
    test_meta[:, 5]      += m.predict_proba(X_te_xgb)[:, 1] / N_FOLDS
    fold_aucs.append(roc_auc_score(y_val, oof_meta[val_idx, 5]))

    print(f'  LGB_v1={fold_aucs[0]:.5f}  CAT_v1={fold_aucs[1]:.5f}  XGB_v1={fold_aucs[2]:.5f}')
    print(f'  LGB_v2={fold_aucs[3]:.5f}  CAT_v2={fold_aucs[4]:.5f}  XGB_v2={fold_aucs[5]:.5f}')

# L1 개별 OOF AUC
print('\n' + '='*60)
for i, name in enumerate(BASE_MODEL_NAMES):
    print(f'  {name:10s} OOF AUC: {roc_auc_score(y_train, oof_meta[:, i]):.5f}')
print('='*60)
print(f'메타 피처 행렬: {oof_meta.shape}  /  test: {test_meta.shape}')


── Fold 1/5 ──────────────────────────────────
  LGB_v1=0.73812  CAT_v1=0.73818  XGB_v1=0.73854
  LGB_v2=0.73824  CAT_v2=0.73827  XGB_v2=0.73833

── Fold 2/5 ──────────────────────────────────
  LGB_v1=0.74318  CAT_v1=0.74312  XGB_v1=0.74340
  LGB_v2=0.74317  CAT_v2=0.74338  XGB_v2=0.74301

── Fold 3/5 ──────────────────────────────────
  LGB_v1=0.74072  CAT_v1=0.74004  XGB_v1=0.74037
  LGB_v2=0.74068  CAT_v2=0.74020  XGB_v2=0.74036

── Fold 4/5 ──────────────────────────────────
  LGB_v1=0.73880  CAT_v1=0.73866  XGB_v1=0.73872
  LGB_v2=0.73871  CAT_v2=0.73864  XGB_v2=0.73893

── Fold 5/5 ──────────────────────────────────
  LGB_v1=0.74160  CAT_v1=0.74077  XGB_v1=0.74158
  LGB_v2=0.74129  CAT_v2=0.74062  XGB_v2=0.74156

  LGB_v1     OOF AUC: 0.74047
  CAT_v1     OOF AUC: 0.74015
  XGB_v1     OOF AUC: 0.74051
  LGB_v2     OOF AUC: 0.74041
  CAT_v2     OOF AUC: 0.74022
  XGB_v2     OOF AUC: 0.74043
메타 피처 행렬: (256351, 6)  /  test: (90067, 6)


## 4. Level-2 Meta 모델 학습

In [5]:
# Meta 모델도 OOF로 학습 (과적합 방지)
oof_meta_pred  = np.zeros(len(y_train))
test_meta_pred = np.zeros(len(X_v1_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(oof_meta, y_train), 1):
    X_tr_m  = oof_meta[tr_idx]
    X_val_m = oof_meta[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    ds_tr  = lgb.Dataset(X_tr_m, label=y_tr,
                          feature_name=BASE_MODEL_NAMES)
    ds_val = lgb.Dataset(X_val_m, label=y_val, reference=ds_tr)
    m = lgb.train(META_PARAMS, ds_tr, num_boost_round=500,
                  valid_sets=[ds_val],
                  callbacks=[lgb.early_stopping(50, verbose=False),
                              lgb.log_evaluation(-1)])
    oof_meta_pred[val_idx]  = m.predict(X_val_m)
    test_meta_pred         += m.predict(test_meta) / N_FOLDS

    print(f'  Fold {fold}  Meta AUC: {roc_auc_score(y_val, oof_meta_pred[val_idx]):.5f}'
          f'  best_iter={m.best_iteration}')

auc_meta = roc_auc_score(y_train, oof_meta_pred)
BASELINE = 0.74071
print(f'\n[Meta L2] OOF AUC: {auc_meta:.5f}  ({auc_meta-BASELINE:+.5f} vs EXP-032)')

# 비교: 단순 rank average
rank_avg_oof = np.stack([rankdata(oof_meta[:, i]) for i in range(N_BASE)], axis=1).mean(axis=1)
auc_rank_avg = roc_auc_score(y_train, rank_avg_oof)
print(f'[Rank Avg L1] OOF AUC: {auc_rank_avg:.5f}  ({auc_rank_avg-BASELINE:+.5f} vs EXP-032)')
print(f'\n→ Stacking 개선: {auc_meta - auc_rank_avg:+.5f}')

  Fold 1  Meta AUC: 0.73840  best_iter=143
  Fold 2  Meta AUC: 0.74335  best_iter=50
  Fold 3  Meta AUC: 0.74052  best_iter=76
  Fold 4  Meta AUC: 0.73893  best_iter=80
  Fold 5  Meta AUC: 0.74138  best_iter=62

[Meta L2] OOF AUC: 0.74002  (-0.00069 vs EXP-032)
[Rank Avg L1] OOF AUC: 0.74073  (+0.00002 vs EXP-032)

→ Stacking 개선: -0.00071


## 5. Submission 저장 & 실험 기록

In [6]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Meta 예측 저장
sub_meta = sub.copy()
sub_meta['probability'] = test_meta_pred
auc_str = f'{auc_meta:.4f}'.replace('.', '')
fname_meta = f'submission_exp{EXP_NO:03d}_{AUTHOR}_stacking_{auc_str}.csv'
sub_meta.to_csv(OUT_DIR / fname_meta, index=False)
print(f'[Stacking] 저장: {fname_meta}')
print(f'  probability 범위: [{test_meta_pred.min():.4f}, {test_meta_pred.max():.4f}]')

# Rank Avg 비교용 저장
rank_avg_test = np.stack([rankdata(test_meta[:, i]) for i in range(N_BASE)], axis=1).mean(axis=1)
rank_avg_test = (rank_avg_test - rank_avg_test.min()) / (rank_avg_test.max() - rank_avg_test.min())
sub_rank = sub.copy()
sub_rank['probability'] = rank_avg_test
auc_str_r = f'{auc_rank_avg:.4f}'.replace('.', '')
fname_rank = f'submission_exp{EXP_NO:03d}_{AUTHOR}_rankavg_{auc_str_r}.csv'
sub_rank.to_csv(OUT_DIR / fname_rank, index=False)
print(f'[Rank Avg] 저장: {fname_rank}')

best_auc  = max(auc_meta, auc_rank_avg)
best_pred = test_meta_pred if auc_meta >= auc_rank_avg else rank_avg_test
best_name = 'Stacking' if auc_meta >= auc_rank_avg else 'Rank Average'
print(f'\n→ 최적: {best_name}  OOF AUC={best_auc:.5f}')

[Stacking] 저장: submission_exp037_조여진_stacking_07400.csv
  probability 범위: [0.0101, 0.7942]
[Rank Avg] 저장: submission_exp037_조여진_rankavg_07407.csv

→ 최적: Rank Average  OOF AUC=0.74073


In [7]:
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

def log_to_leaderboard(exp_no, author, model_name, params_str,
                        f1, recall, precision, accuracy, oof_auc,
                        cv_strategy, preprocessing_ver, n_features,
                        imbalance_method, submitted, hackathon_score,
                        file_name, notes='', insights=''):
    lb_path = DOCS_DIR / 'leaderboard.xlsx'
    wb = load_workbook(lb_path)
    ws = wb['리더보드']
    exp_label = f'EXP-{exp_no:03d}'
    next_row = ws.max_row + 1
    for r in range(2, ws.max_row + 1):
        val = ws.cell(row=r, column=2).value
        if val == exp_label:
            next_row = r; break
        if ws.cell(row=r, column=1).value is None or str(ws.cell(row=r, column=1).value).strip() == '':
            next_row = r; break
    values = [str(date.today()), exp_label, author, model_name, params_str,
              round(f1,5), round(recall,5), round(precision,5), round(accuracy,5), round(oof_auc,5),
              cv_strategy, preprocessing_ver, n_features, imbalance_method,
              submitted, hackathon_score, file_name, notes, insights]
    thin   = Side(style='thin', color='B0B8D0')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    fill   = PatternFill('solid', fgColor='EEF2FA') if next_row % 2 == 0 else None
    font   = Font(name='맑은 고딕', size=10)
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left   = Alignment(horizontal='left',   vertical='center', wrap_text=True)
    left_cols = {4, 5, 12, 14, 17, 18, 19}
    for c_idx, val in enumerate(values, start=1):
        cell = ws.cell(row=next_row, column=c_idx, value=val)
        cell.font = font; cell.border = border
        cell.alignment = left if c_idx in left_cols else center
        if fill: cell.fill = fill
        if c_idx in range(6, 11) or c_idx == 16: cell.number_format = '0.00000'
    wb.save(lb_path)
    print(f'[leaderboard.xlsx] EXP-{exp_no:03d} 기록 완료 (row {next_row})')

oof_binary = (oof_meta_pred >= 0.5).astype(int)
params_str = (f'L1: LGB+CAT+XGB(FE-v1) + LGB+CAT+XGB(FE-v2+TE+ITE) | '
              f'L2: LGB(leaves=8,depth=3) | best={best_name}')
NOTES    = ('Stacking: 6개 Base 모델 OOF → LGB Meta 학습. '
            'FE-v1 3개 + FE-v2+TE+ITE 3개. '
            'Meta LGB는 과적합 방지 위해 shallow (leaves=8, depth=3).')
INSIGHTS = (f'EXP-032 대비 {best_auc-BASELINE:+.5f} | '
            f'Stacking={auc_meta:.5f} vs RankAvg(6모델)={auc_rank_avg:.5f} | '
            f'Stacking 개선: {auc_meta-auc_rank_avg:+.5f}')

log_to_leaderboard(
    EXP_NO, AUTHOR, 'Stacking(LGB+CAT+XGB ×2 → LGB-meta)', params_str,
    f1_score(y_train, oof_binary), recall_score(y_train, oof_binary),
    precision_score(y_train, oof_binary), accuracy_score(y_train, oof_binary),
    best_auc, CV_STR, 'mixed(v1/v2+TE+ITE)', X_v1_train.shape[1],
    'is_unbalance / Balanced / scale_pos_weight',
    'N', None, 'notebooks/33_stacking_v2_yjcho.ipynb', NOTES, INSIGHTS
)

[leaderboard.xlsx] EXP-037 기록 완료 (row 34)
